# Anotação dos clusters
Neste arquivo é feita a anotação dos clusters para treinamento

In [1]:
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from datetime import datetime, timedelta

### 1. Importação e tabulação

In [2]:
# Importação dados OR
df_or = gpd.read_file('dados/remocoes_2010-2025.geojson').to_crs('EPSG:31983').query('categoria != "Ameaça de remoção" & data_remocao != "desconhecida"')
df_or = df_or[df_or['data_remocao'].apply(lambda x: len(x)==7)]

# Importação dados TJ
df_tj = gpd.read_file('dados/posse_2025.geojson').to_crs('EPSG:31983')
df_tj = df_tj[df_tj['Sentença'].fillna('').apply(lambda x: 'Procedente' in x)]

# Importação clusters
df_mudancas = gpd.read_parquet('clusters/origem_deslocamentos_clusterizados.parquet').to_crs('EPSG:31983')
df_mudancas['date_range'] = df_mudancas.apply(lambda x: (x['DT_ATUALIZACAO_FAM'], x['DT_ATUALIZACAO_FAM_pos']), axis=1)

c:\Users\Pedro\anaconda3\envs\geo\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


#### Agregados por cluster

In [3]:
df_clusters = gpd.GeoDataFrame()

cluster_cols = [i for i in df_mudancas.columns if i.startswith('cluster')]
for c in cluster_cols:
    print(f'Concatenando {c}')
    df_temp = df_mudancas.pivot_table(index=c, 
                                        values=['LATITUDE', 'LONGITUDE', 'CO_FAMILIAR_FAM', 'date_range', 'data_media'], 
                                        aggfunc={'LATITUDE': 'mean', 'LONGITUDE': 'mean', 'CO_FAMILIAR_FAM': 'count', 'date_range': lambda x: list(x), 'data_media':'mean'}, 
                                        fill_value=0).reset_index().rename(columns={'CO_FAMILIAR_FAM':'num_familias'})

    df_temp['cluster_code'] = df_temp[c].apply(lambda x: '.'.join(f'{c}_{x:05d}'.split('_')[1:]) if x>=0 else None)
    df_temp = df_temp.rename(columns={'CO_FAMILIAR_FAM':'num_familias', c:'cluster_id'})
    df_temp['cluster_group'] = c
    df_temp = gpd.GeoDataFrame(df_temp, 
                               geometry=gpd.GeoSeries.from_xy(df_temp['LONGITUDE'], df_temp['LATITUDE']), 
                               crs='EPSG:4326').to_crs('EPSG:31983').query(f'num_familias >=10 & cluster_id>=0').drop(['LATITUDE', 'LONGITUDE'], axis=1)
    df_clusters = pd.concat([df_clusters, df_temp])

Concatenando cluster_30m_6M
Concatenando cluster_60m_6M
Concatenando cluster_30m_12M
Concatenando cluster_60m_12M
Concatenando cluster_100m_6M
Concatenando cluster_100m_12M


#### Indexação em KDTrees para otimização do alinhamento

In [4]:
# Indexação em KDTrees
or_tree = cKDTree(df_or['geometry'].apply(lambda x: (x.x, x.y)).to_list())
tj_tree = cKDTree(df_tj['geometry'].apply(lambda x: (x.x, x.y)).to_list())
clusters_tree = cKDTree(df_clusters['geometry'].apply(lambda x: (x.x, x.y)).to_list())

In [5]:
# Listagem das remoções (OR e TJ) a 500 metros do centro de cada cluster
df_clusters['pontos_or'] = clusters_tree.query_ball_tree(or_tree, r=500)
df_clusters['pontos_tj'] = clusters_tree.query_ball_tree(tj_tree, r=500)

df_clusters['candidates_count'] = df_clusters['pontos_or'].apply(lambda x: len(x)) + df_clusters['pontos_tj'].apply(lambda x: len(x))

#### Dataframe de mudanças com dados agregados dos clusters

In [6]:
df_cluster_points = df_mudancas.melt(id_vars=['id_mudanca'], value_vars=cluster_cols, var_name='cluster_group', value_name='cluster_id').query('cluster_id >= 0')
df_cluster_points = df_cluster_points.merge(df_clusters, on=['cluster_group', 'cluster_id'], how='left', suffixes=('', '_cluster'))

datacols = ['CO_FAMILIAR_FAM', 'DT_ATUALIZACAO_FAM', 'data_media', 'geometry', 'geometry_pos',
       'distancia', 'linha_deslocamento', 'date_mean', 'date_range', 'id_mudanca']
df_cluster_points = df_cluster_points.merge(df_mudancas[[i for i in df_mudancas.columns if i not in cluster_cols]], on='id_mudanca', how='left', suffixes=('_cluster', ''))

df_cluster_points = gpd.GeoDataFrame(df_cluster_points, geometry='geometry', crs='EPSG:31983')

In [7]:
print('Clusters totais:', len(df_cluster_points['cluster_code'].unique()))
print('Clusters com potencial de alinhamento:', len(df_cluster_points[df_cluster_points['candidates_count']>0]['cluster_code'].unique()))

Clusters totais: 64193
Clusters com potencial de alinhamento: 21959


### 2. Cálculo dos scores

Critérios de seleção:
- **pct_date_subset**: percentual de mudanças cujos registros contém o período de seleção (data da remoção +30 dias para remoções OR, ano de abertura a data disponibilização para remoções TJ)
- **pct_date_overlap**: percentual de mudanças cujos registros interseccionam com o período de seleção (data da remoção +30 dias para remoções OR, ano de abertura a data disponibilização para remoções TJ)

Critérios para validação e desempate:
- **median_timesprawl**: mediana da diferença entre a data média do cluster e a data média de cada registro
- **median_geosprawl**: mediana da distância entre o centroide do cluster e o ponto de origem da mudança
- **mean_timesprawl**: média da diferença entre a data média do cluster e a data média de cada registro
- **mean_geosprawl**: média da distância entre o centroide do cluster e o ponto de origem da mudança

In [8]:
df_cluster_points.columns

Index(['id_mudanca', 'cluster_group', 'cluster_id', 'num_familias',
       'data_media_cluster', 'date_range_cluster', 'cluster_code',
       'geometry_cluster', 'pontos_or', 'pontos_tj', 'candidates_count',
       'CO_FAMILIAR_FAM', 'DT_ATUALIZACAO_FAM', 'DT_ATUALIZACAO_FAM_pos',
       'data_media', 'LATITUDE', 'LONGITUDE', 'LATITUDE_pos', 'LONGITUDE_pos',
       'geometry', 'geometry_pos', 'distancia', 'linha_deslocamento',
       'date_range'],
      dtype='object')

#### Cálculo dos scores de espalhamento espacial e temporal

In [9]:
df_cluster_points['geosprawl'] = df_cluster_points['geometry'].distance(df_cluster_points['geometry_cluster'])
df_cluster_points['timesprawl'] = abs(df_cluster_points['data_media_cluster']-df_cluster_points['data_media'])

In [10]:
df_sprawlscores = df_cluster_points.pivot_table(index=['cluster_code'], values=['geosprawl', 'timesprawl'], aggfunc=['mean', 'median']).reset_index()
df_sprawlscores.columns = ['_'.join([j for j in i if j]) for i in df_sprawlscores.columns]

#### Cálculo dos scores de sobreposição temporal

In [11]:
def calc_date_subset(interval, subset):
    return 1 if (interval[0] <= subset[0] and interval[1] >= subset[1]) else 0

def calc_date_overlap(interval, overlap):
    return 1 if max(interval[0], overlap[0]) < min(interval[1], overlap[1]) else 0

In [ ]:
df_clusters_or = df_cluster_points.explode('pontos_or').dropna(subset='pontos_or')
df_clusters_or['data_remocao'] = df_clusters_or['pontos_or'].apply(lambda x: datetime.strptime(df_or.iloc[x]['data_remocao'], '%m/%Y'))
df_clusters_or['date_subset'] = df_clusters_or.apply(lambda x: calc_date_subset(x['date_range'], [x['data_remocao'], x['data_remocao']+timedelta(days=30)]), axis=1)
df_clusters_or['date_overlap'] = df_clusters_or.apply(lambda x: calc_date_overlap(x['date_range'], [x['data_remocao'], x['data_remocao']+timedelta(days=30)]), axis=1)

df_clusters_or = df_clusters_or.pivot_table(index=['cluster_code', 'pontos_or'], 
                                            values=['date_subset', 'date_overlap', 'id_mudanca'],
                                            aggfunc={'date_subset':'sum', 'date_overlap':'sum', 'id_mudanca':'count'}).reset_index()

df_clusters_or = df_clusters_or.rename(columns={'id_mudanca':'num_points'})
df_clusters_or['pct_date_subset'] = df_clusters_or['date_subset']/df_clusters_or['num_points']
df_clusters_or['pct_date_overlap'] = df_clusters_or['date_overlap']/df_clusters_or['num_points']
df_clusters_or['tipo'] = 'OR'
df_clusters_or = df_clusters_or.rename(columns={'pontos_or':'remocao_id'})

In [ ]:
df_clusters_tj = df_cluster_points.explode('pontos_tj').dropna(subset='pontos_tj')
df_clusters_tj['data_abertura'] = df_clusters_tj['pontos_tj'].apply(lambda x: datetime.strptime(str(df_tj.iloc[x]['Ano etiqueta']), '%Y'))
df_clusters_tj['data_sentenca'] = df_clusters_tj['pontos_tj'].apply(lambda x: datetime.strptime(df_tj.iloc[x]['Data de Disponibilização'], '%d/%m/%Y'))
df_clusters_tj['date_subset'] = df_clusters_tj.apply(lambda x: calc_date_subset(x['date_range'], [x['data_abertura'], x['data_sentenca']]), axis=1)
df_clusters_tj['date_overlap'] = df_clusters_tj.apply(lambda x: calc_date_overlap(x['date_range'], [x['data_abertura'], x['data_sentenca']]), axis=1)

df_clusters_tj = df_clusters_tj.pivot_table(index=['cluster_code', 'pontos_tj'], 
                                            values=['date_subset', 'date_overlap', 'id_mudanca'],
                                            aggfunc={'date_subset':'sum', 'date_overlap':'sum', 'id_mudanca':'count'}).reset_index()

df_clusters_tj = df_clusters_tj.rename(columns={'id_mudanca':'num_points'})
df_clusters_tj['pct_date_subset'] = df_clusters_tj['date_subset']/df_clusters_tj['num_points']
df_clusters_tj['pct_date_overlap'] = df_clusters_tj['date_overlap']/df_clusters_tj['num_points']
df_clusters_tj['tipo'] = 'TJ'
df_clusters_tj = df_clusters_tj.rename(columns={'pontos_tj':'remocao_id'})

#### Junção dos scores

In [ ]:
df_scores = pd.concat([df_clusters_tj, df_clusters_or])
df_scores = df_scores.merge(df_sprawlscores, on='cluster_code', how='left')
df_scores.to_parquet('clusters/scores/cluster_scores.parquet', engine='pyarrow')

### 3. Definição das anotações

Consideramos como positivos os pares remoção-clusters classificados como "best match", e como negativos os pares "non viable match"

In [47]:
df_scores = pd.read_parquet('clusters/scores/cluster_scores.parquet')
df_scores = df_scores.sort_values(by=['cluster_code', 'pct_date_overlap', 'pct_date_subset', 'median_timesprawl', 'median_geosprawl'], ascending=False)

for c in ['mean_timesprawl', 'median_timesprawl']:
    df_scores[c] = df_scores[c].apply(lambda x: x.days)

In [54]:
# Aplicação de critérios de seleção
df_scores['candidate'] = (df_scores['pct_date_overlap']>0.5) & (df_scores['median_timesprawl'] < 200)
df_scores = df_scores.merge(df_scores.pivot_table(index='cluster_code', 
                                                  values='candidate', 
                                                  aggfunc='max').reset_index().rename(columns={'candidate':'cluster_is_candidate'}),
                            on='cluster_code', how='left')

In [58]:
df_scores['cluster_ranking'] = df_scores.groupby('cluster_code').cumcount(ascending=True) +1

In [79]:
def classify_candidate(x):
    if x['candidate'] and x['cluster_is_candidate'] and x['cluster_ranking']==1:
        return '0_best match'
    elif x['candidate'] and x['cluster_is_candidate'] and x['cluster_ranking']==2:
        return '1_viable match'
    elif x['candidate'] and x['cluster_is_candidate'] and x['cluster_ranking']>2:
            return '2_non viable match'
    else:
        return '3_unknown'

df_scores['classification'] = df_scores.apply(classify_candidate, axis=1)
df_scores.to_parquet('clusters/cluster_match_classification.parquet', engine='pyarrow')

In [78]:
df_scores.pivot_table(index='classification', values='cluster_code', aggfunc='count').reset_index()

,classification,cluster_code
0,0_best match,5719
1,1_viable match,957
2,2_non viable match,584
3,3_unknown,29004
